# 03 — Tool Calling

Tool calling (function calling) lets a model decide when to invoke external functions
rather than generating an answer from its parameters alone. The model returns a structured
request; your code executes the function and feeds the result back; the model produces
the final response.

This notebook tests **3 tools × 3 scenarios × 3 local models**:

| Tool | What it does |
| --- | --- |
| `get_headcount(department)` | Returns headcount for a Contoso department |
| `get_attrition_rate(quarter)` | Returns attrition % for a given quarter |
| `calculate_salary_budget(department, raise_pct)` | Computes new budget after a raise |

| Scenario | Description |
| --- | --- |
| 1 — Single call | Question answered by one tool |
| 2 — Sequential calls | Two tools needed, results combined |
| 3 — Refusal | Out-of-scope question — model should answer directly, no tool |

> **Ollama API**: pass Python functions directly via `tools=[...]`. Schemas are
> generated automatically from Google-style docstrings and type hints.

In [1]:
from utils.helpers import check_model_available, check_ollama_running

# ── Models ──────────────────────────────────────────────────────────────────
PRIMARY_MODEL = "qwen3.5:9b"        # best tool-calling support among local models
ALL_MODELS = ["mistral:7b", "qwen3.5:9b", "gemma4:e4b"]

if not check_ollama_running():
    raise RuntimeError("Ollama is not running. Start it with: ollama serve")

available = [m for m in ALL_MODELS if check_model_available(m)]
missing = [m for m in ALL_MODELS if m not in available]
print(f"Available models: {available}")
if missing:
    print(f"Missing (skipped): {missing}")

Available models: ['mistral:7b', 'qwen3.5:9b', 'gemma4:e4b']


## 1. Tool definitions

Each function has a Google-style docstring. Ollama reads the function signature and
docstring to build the JSON schema it sends to the model — no manual schema writing needed.

In [2]:
import json

# ── Data backing the tools (sourced from contoso_report_q4_2024.md) ─────────
_HEADCOUNTS = {
    "engineering": 72,
    "sales": 38,
    "customer success": 31,
    "product": 22,
    "marketing": 19,
    "finance & operations": 17,
    "people & hr": 9,
    "executive": 6,
}

_ATTRITION = {"Q1": 3.1, "Q2": 4.0, "Q3": 5.3, "Q4": 4.2, "ANNUAL": 14.8}

_AVG_SALARIES = {
    "engineering": 91_200,
    "product": 87_400,
    "executive": 212_000,
    "finance & operations": 74_100,
    "sales": 69_800,
    "customer success": 58_300,
    "marketing": 62_100,
    "people & hr": 55_900,
}


# ── Tools ───────────────────────────────────────────────────────────────────

def get_headcount(department: str) -> int:
    """Return the headcount for a Contoso Corp department as of Q4 2024.

    Args:
        department (str): Department name, e.g. 'Engineering', 'Sales', 'Marketing'.

    Returns:
        int: Number of employees, or -1 if the department is not found.
    """
    return _HEADCOUNTS.get(department.lower().strip(), -1)


def get_attrition_rate(quarter: str) -> float:
    """Return the employee attrition rate for a given quarter at Contoso Corp.

    Args:
        quarter (str): Quarter identifier: 'Q1', 'Q2', 'Q3', 'Q4', or 'annual'.

    Returns:
        float: Attrition rate as a percentage, or -1.0 if the quarter is unknown.
    """
    return _ATTRITION.get(quarter.upper().strip(), -1.0)


def calculate_salary_budget(department: str, raise_pct: float) -> str:
    """Calculate the new total salary budget for a department after a percentage raise.

    Args:
        department (str): Department name, e.g. 'Engineering', 'Marketing'.
        raise_pct (float): Raise percentage to apply, e.g. 5.0 for a 5 % raise.

    Returns:
        str: JSON string with current_budget_gbp, new_budget_gbp, increase_gbp.
    """
    key = department.lower().strip()
    if key not in _HEADCOUNTS or key not in _AVG_SALARIES:
        return json.dumps({"error": f"Department '{department}' not found"})

    headcount = _HEADCOUNTS[key]
    avg_salary = _AVG_SALARIES[key]
    current = headcount * avg_salary
    new = round(current * (1 + raise_pct / 100))
    return json.dumps({
        "department": department,
        "headcount": headcount,
        "avg_salary_gbp": avg_salary,
        "current_budget_gbp": current,
        "new_budget_gbp": new,
        "increase_gbp": new - current,
    })


TOOLS = [get_headcount, get_attrition_rate, calculate_salary_budget]
AVAILABLE_FUNCTIONS = {f.__name__: f for f in TOOLS}

print("Tools registered:")
for f in TOOLS:
    print(f"  • {f.__name__}")

Tools registered:
  • get_headcount
  • get_attrition_rate
  • calculate_salary_budget


In [3]:
import time

from ollama import chat


def run_with_tools(
    model: str,
    user_message: str,
    tools: list,
    max_turns: int = 6,
    verbose: bool = False,
) -> dict:
    """Agentic tool-calling loop: call model, execute tools, repeat until done.

    Args:
        model: Ollama model name.
        user_message: Initial user question.
        tools: List of Python functions to expose as tools.
        max_turns: Safety limit to prevent infinite loops.
        verbose: Print each step when True.

    Returns:
        Dict with final_response, tool_calls (list), turns, elapsed_s.
    """
    messages = [{"role": "user", "content": user_message}]
    tool_calls_log: list[dict] = []
    t0 = time.time()

    for turn in range(max_turns):
        response = chat(model=model, messages=messages, tools=tools)

        if not response.message.tool_calls:
            return {
                "final_response": response.message.content,
                "tool_calls": tool_calls_log,
                "turns": turn + 1,
                "elapsed_s": round(time.time() - t0, 1),
            }

        # Append assistant message (contains the tool call requests)
        messages.append(response.message)

        for tc in response.message.tool_calls:
            name = tc.function.name
            args = tc.function.arguments
            result = AVAILABLE_FUNCTIONS[name](**args)
            tool_calls_log.append({"function": name, "args": args, "result": result})

            if verbose:
                print(f"  → {name}({args}) = {result}")

            messages.append({
                "role": "tool",
                "content": str(result),
                "tool_name": name,
            })

    return {
        "final_response": "(max turns reached)",
        "tool_calls": tool_calls_log,
        "turns": max_turns,
        "elapsed_s": round(time.time() - t0, 1),
    }


print("run_with_tools() helper ready")

run_with_tools() helper ready


## 2. Scenario 1 — Single tool call

The model needs exactly one tool to answer. We walk through the interaction
step by step before running the comparison loop.

In [4]:
Q1 = "How many people work in the Engineering department at Contoso Corp?"

result = run_with_tools(PRIMARY_MODEL, Q1, TOOLS, verbose=True)

print(f"\nQuestion:  {Q1}")
print(f"Tools used: {[tc['function'] for tc in result['tool_calls']]}")
print(f"Answer:    {result['final_response'].strip()}")
print(f"Time:      {result['elapsed_s']}s | Turns: {result['turns']}")

  → get_headcount({'department': 'Engineering'}) = 72

Question:  How many people work in the Engineering department at Contoso Corp?
Tools used: ['get_headcount']
Answer:    There are 72 people working in the Engineering department at Contoso Corp as of Q4 2024.
Time:      13.8s | Turns: 2


## 3. Scenario 2 — Sequential tool calls

The question requires two different tools. The model may call them in a single
turn (parallel) or sequentially across two turns — both are valid.

In [5]:
Q2 = (
    "What was the Q4 2024 attrition rate at Contoso? "
    "Also, what would the Engineering department's total salary budget look like "
    "after a 10 % raise?"
)

result2 = run_with_tools(PRIMARY_MODEL, Q2, TOOLS, verbose=True)

print(f"\nTools called: {[tc['function'] for tc in result2['tool_calls']]}")
print(f"\nAnswer:\n{result2['final_response'].strip()}")
print(f"\nTime: {result2['elapsed_s']}s | Turns: {result2['turns']}")

  → get_attrition_rate({'quarter': 'Q4'}) = 4.2
  → calculate_salary_budget({'department': 'Engineering', 'raise_pct': 10}) = {"department": "Engineering", "headcount": 72, "avg_salary_gbp": 91200, "current_budget_gbp": 6566400, "new_budget_gbp": 7223040, "increase_gbp": 656640}

Tools called: ['get_attrition_rate', 'calculate_salary_budget']

Answer:
Here are the results for your questions:

**Q4 2024 Attrition Rate:**
The employee attrition rate at Contoso Corp for Q4 2024 was **4.2%**.

**Engineering Department Salary Budget (after 10% raise):**
- **Headcount:** 72 employees
- **Average Salary:** £91,200 GBP
- **Current Budget:** £6,566,400 GBP
- **New Budget (after 10% raise):** £7,223,040 GBP
- **Total Increase:** £656,640 GBP

The Engineering department's total salary budget would increase to approximately **£7.22 million** after implementing a 10% raise across the department.

Time: 21.9s | Turns: 2


## 4. Scenario 3 — Out-of-scope question (no tool should be called)

A well-behaved model recognises when none of the available tools are relevant
and answers from general knowledge. Calling a tool for an unrelated question
is a hallucination of a different kind.

In [6]:
Q3 = "What is the capital of France?"

result3 = run_with_tools(PRIMARY_MODEL, Q3, TOOLS, verbose=True)

tool_names = [tc["function"] for tc in result3["tool_calls"]]
no_tool_called = len(tool_names) == 0

print(f"Question: {Q3}")
print(f"Tools called: {tool_names if tool_names else 'none'}")
print(f"Answer: {result3['final_response'].strip()}")
label = '✓ correct' if no_tool_called else '✗ called a tool unnecessarily'
print(f"\nExpected behaviour (no tool): {label}")

Question: What is the capital of France?
Tools called: none
Answer: The capital of France is Paris. It is located in the north-central part of the country and is France's largest city, serving as its political, cultural, and economic center.

Expected behaviour (no tool): ✓ correct


## 5. Full comparison — 3 models × 3 scenarios

Success criteria per scenario:

| Scenario | Success condition |
| --- | --- |
| 1 — Single | `get_headcount` called, result = 72 |
| 2 — Sequential | Both `get_attrition_rate` and `calculate_salary_budget` called |
| 3 — Refusal | Zero tools called |

In [7]:
SCENARIOS = [
    ("S1 — Single", Q1),
    ("S2 — Sequential", Q2),
    ("S3 — Refusal", Q3),
]


def score_scenario(scenario_label: str, result: dict) -> bool:
    """Return True if the model behaved correctly for this scenario."""
    calls = result["tool_calls"]
    names = [c["function"] for c in calls]

    if scenario_label.startswith("S1"):
        return (
            "get_headcount" in names
            and any(
                c["result"] == 72
                for c in calls
                if c["function"] == "get_headcount"
            )
        )
    if scenario_label.startswith("S2"):
        return (
            "get_attrition_rate" in names
            and "calculate_salary_budget" in names
        )
    if scenario_label.startswith("S3"):
        return len(names) == 0
    return False


# ── Run all combinations ────────────────────────────────────────────────────
rows = []
for model in available:
    row = {"model": model}
    score = 0
    for label, question in SCENARIOS:
        try:
            r = run_with_tools(model, question, TOOLS)
            ok = score_scenario(label, r)
        except Exception:
            ok = False
        row[label] = ok
        score += int(ok)
    row["rate"] = f"{score}/{len(SCENARIOS)}"
    rows.append(row)

# ── Display table ───────────────────────────────────────────────────────────
col_labels = [label for label, _ in SCENARIOS]
header = f"{'Model':<22}" + "".join(f"  {lbl:<18}" for lbl in col_labels) + "  Rate"
print(header)
print("─" * len(header))
for row in rows:
    line = f"{row['model']:<22}"
    for label in col_labels:
        mark = "✓" if row[label] else "✗"
        line += f"  {mark:<18}"
    line += f"  {row['rate']}"
    print(line)

Model                   S1 — Single         S2 — Sequential     S3 — Refusal        Rate
────────────────────────────────────────────────────────────────────────────────────────
mistral:7b              ✗                   ✓                   ✓                   2/3
qwen3.5:9b              ✓                   ✓                   ✓                   3/3
gemma4:e4b              ✓                   ✓                   ✓                   3/3
